# מעבדה 06 — תהליכים תרמודינמיים

במעבדה הזו תריצו גז אידיאלי לאורך כל אחת מארבע המשפחות הכמו-סטטיות, אחר כך תעמידו זו מול זו
שלוש התפשטויות *אדיאבטיות* — איטית, כנגד עומס קבוע, וחופשית אל תוך ריק — ולבסוף תתבוננו
בהתפשטות חופשית המתרחשת רמה אחת למטה, חלקיק אחר חלקיק.

המודול נשען על טענה אחת: אדיאבטי וכמו-סטטי הם תנאים בלתי תלויים, והחוק
$P V^{\gamma} = \text{constant}$ זקוק לשניהם. כל מה שלהלן בנוי כדי להפוך את הטענה הזו
לניתנת להפרכה ולא רק לזכירה.

## מפרט המודל

| | |
|---|---|
| **מערכת** | כמות קבועה של גז אידיאלי, $N$ חלקיקים ולכל אחד $f$ דרגות חופש ריבועיות |
| **דינמיקה** | שני משטרים: כמו-סטטי (זוג $(P,V,T)$ יחיד לאורך כל הדרך) ובלתי הפיך (רק נקודות הקצה הן מצבי שיווי משקל) |
| **גבול** | בוכנה נטולת חיכוך בדופן שהיא דיאתרמית או אדיאבטית; המחיצה של ההתפשטות החופשית אדיאבטית ואינה מבצעת עבודה |
| **צבר** | אינו רלוונטי לתרמודינמיקה; ההתפשטות החופשית המיקרוסקופית היא מיקרו-קנונית ומשמרת אנרגיה במדויק |
| **מוזנח** | חיכוך, מסת הבוכנה והאינרציה שלה, אי-אידיאליות של הגז, דליפת חום דרך דופן אדיאבטית, זמן החזרה לשיווי משקל |
| **תקף כאשר** | תוצאות כמו-סטטיות: איטי ביחס לזמן הרלקסציה. תוצאות בלתי הפיכות: הלחץ החיצוני ידוע ואחיד על הגבול |
| **אופני כישלון** | גזים ממשיים (שההתפשטות החופשית שלהם *כן* מקררת), דחיסות מהירות מכדי שאפילו $P_{\mathrm{ext}}$ יהיה אחיד, דופן דולפת |

מוסכמת הסימנים לאורך כל הדרך, ללא יוצא מן הכלל:
$$ dU = \delta Q + \delta W_{\mathrm{on}}, \qquad \delta W_{\mathrm{on}} = -P_{\mathrm{ext}}\,dV. $$

שימו לב איזה לחץ יושב במשוואה השנייה. זהו הלחץ הפועל על הגבול, והוא שווה ל-$P$ של הגז עצמו
רק כאשר התהליך כמו-סטטי.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import kinetics, processes
from thermolab.validation import relative_error, seed_study

N_PARTICLES = 1000
TEMPERATURE = 300.0  # K
V1 = 1.0e-3  # m^3
VOLUME_RATIO = 2.0
V2 = VOLUME_RATIO * V1

start = processes.EquilibriumState.from_temperature(N_PARTICLES, TEMPERATURE, V1)

c_v = processes.heat_capacity_constant_volume(N_PARTICLES)
c_p = processes.heat_capacity_constant_pressure(N_PARTICLES)

print(f"start   N = {N_PARTICLES}   T = {start.temperature:.1f} K   V = {V1 * 1e3:.2f} L")
print(f"        P = {start.pressure:.4e} Pa    U = {start.internal_energy:.4e} J")
print(f"        f = {start.degrees_of_freedom}   gamma = {start.gamma:.4f}")
print(f"        C_V = {c_v:.4e} J/K    C_P = {c_p:.4e} J/K")
print(f"        C_P - C_V = {c_p - c_v:.4e} J/K   (Mayer's relation says N k_B)")

## חלק 1 — ארבע המשפחות

כל משפחה מקבעת דבר אחד ומשאירה לחוק הראשון להכריע את השאר. לפני הרצת התא הבא, נבאו עבור כל
אחת מן הארבע מי מבין $W_{\mathrm{on}}$, $Q$ ו-$\Delta U$ מתאפס:

- **איזוכורי** — קררו את הגז בנפח קבוע עד שלחצו יורד פי שניים,
- **איזוברי** — הרחיבו ל-$2V_1$ בלחץ קבוע,
- **איזותרמי** — הרחיבו ל-$2V_1$ בטמפרטורה קבועה,
- **אדיאבטי** — הרחיבו ל-$2V_1$ כשזרימת החום סגורה.

בדיוק שלושה מתוך שנים-עשר הערכים מתאפסים. אילו שלושה?

**הניבוי שלכם:**

*(כתבו כאן לפני הרצת התא הבא)*

In [ ]:
families = [
    processes.isochoric(start, 0.5 * start.pressure),
    processes.isobaric(start, V2),
    processes.isothermal(start, V2),
    processes.adiabatic(start, V2),
]

header = f"{'process':<12}{'W_on (J)':>13}{'Q (J)':>13}{'dU (J)':>13}{'T2 (K)':>9}{'residual':>12}"
print(header)
print("-" * len(header))
for family in families:
    print(
        f"{family.label:<12}{family.work_on_gas:>13.4e}{family.heat:>13.4e}"
        f"{family.internal_energy_change:>13.4e}{family.end.temperature:>9.1f}"
        f"{family.first_law_residual:>12.2e}"
    )

כדאי לעצור על העמודה האחרונה. בשורה האיזוברית העבודה הגיעה מ-$-P\Delta V$ והחום מ-$C_P\Delta T$
— שתי נוסחאות שאף אחת מהן אינה יודעת על השנייה. סגירת החוק הראשון לאפס שם היא לפיכך מבחן חי
של יחס מאייר $C_P - C_V = N k_B$: כל ערך אחר עבור $C_P$ והעמודה הזו לא הייתה מתאפסת.

כעת התבוננו בצורות.

In [ ]:
colours = {"isochoric": "#2563eb", "isobaric": "#d97706",
           "isothermal": "#059669", "adiabatic": "#dc2626"}

fig, ax = plt.subplots(figsize=(7, 4.8))
for family in families:
    path = family.quasistatic_path
    ax.plot(path.volumes * 1e3, path.pressures, lw=2.2,
            color=colours[family.label], label=family.label)
ax.plot([V1 * 1e3], [start.pressure], "ko", ms=8, zorder=5)
ax.set_xlabel("volume (L)")
ax.set_ylabel("pressure (Pa)")
ax.set_ylim(0, 1.12 * start.pressure)
ax.set_title("four families, one starting state")
ax.legend()
plt.tight_layout()
plt.show()

isotherm, adiabat = families[2], families[3]
print(f"final pressure on the isotherm: {isotherm.end.pressure:.4e} Pa")
print(f"final pressure on the adiabat:  {adiabat.end.pressure:.4e} Pa")
print(f"ratio: {adiabat.end.pressure / isotherm.end.pressure:.4f}"
      f"   (theory: 2^(1-gamma) = {2.0 ** (1 - start.gamma):.4f})")

## חלק 2 — שלוש התפשטויות אדיאבטיות

כעת החידה מעמוד המודול. שלוש דרכים, כולן עם $Q = 0$, כולן מאותו מצב אל אותו נפח סופי:

1. **איטית.** הסירו את העומס גרגר אחר גרגר, כך שהגז בשיווי משקל לאורך כל הדרך.
2. **כנגד עומס קבוע.** שחררו את הבוכנה כנגד לחץ חיצוני קבוע, שנבחר כך שהגז נעצר בדיוק
   ב-$2V_1$.
3. **חופשית.** משכו את המחיצה; הגז מתפשט אל תוך ריק.

רק הראשונה כמו-סטטית. התבוננו במחיר של זה.

In [ ]:
p_external = processes.external_pressure_for_equilibrium_at(start, VOLUME_RATIO)
routes = [
    ("slow (quasistatic)", processes.adiabatic(start, V2)),
    ("against a fixed load", processes.adiabatic_against_constant_pressure(start, p_external)),
    ("free expansion", processes.free_expansion(start, V2)),
]

scale = start.pressure * V1
print(f"external load on route 2: {p_external:.4e} Pa = {p_external / start.pressure:.3f} P1\n")
header = (f"{'route':<22}{'T2 (K)':>9}{'P2/P1':>9}{'V2/V1':>8}"
          f"{'W_on (J)':>13}{'-W_on/P1V1':>12}{'Q (J)':>8}")
print(header)
print("-" * len(header))
for name, route in routes:
    print(
        f"{name:<22}{route.end.temperature:>9.1f}"
        f"{route.end.pressure / start.pressure:>9.4f}{route.end.volume / V1:>8.3f}"
        f"{route.work_on_gas:>13.4e}{-route.work_on_gas / scale:>12.3f}{route.heat:>8.1f}"
    )

slow, loaded, free = (route for _, route in routes)
print(f"\nthe slow route delivers "
      f"{slow.work_on_gas / loaded.work_on_gas:.3f} times the work of the loaded one")
print(f"free expansion: dU/U = {free.internal_energy_change / start.internal_energy:.2e}"
      f"   (floating-point zero), so dT = {free.temperature_change:.2e} K")

In [ ]:
fig, (plane, thermal) = plt.subplots(1, 2, figsize=(10.5, 4.2))

slow_path = slow.quasistatic_path
plane.plot(slow_path.volumes * 1e3, slow_path.pressures, lw=2.2, color="#dc2626")
plane.fill_between(slow_path.volumes * 1e3, slow_path.pressures, alpha=0.20, color="#dc2626")
sweep = np.linspace(V1, V2, 200)
plane.axhline(p_external, color="#d97706", lw=1.4, ls=":")
plane.fill_between(sweep * 1e3, p_external, alpha=0.22, color="#d97706")
plane.plot([V1 * 1e3], [start.pressure], "ko", ms=8, zorder=6)
for route, colour in zip(
    (slow, loaded, free), ("#dc2626", "#d97706", "#059669"), strict=True
):
    plane.plot([V2 * 1e3], [route.end.pressure], "o", ms=8, color=colour, zorder=6)
plane.set_xlabel("volume (L)")
plane.set_ylabel("pressure (Pa)")
plane.set_ylim(0, 1.12 * start.pressure)
plane.set_title("shaded area = work delivered")

# Only the quasistatic route has states in between, so only it gets a curve. The other two
# are two dots with nothing drawn between them, which is what "no path" looks like.
volumes = np.linspace(V1, V2, 200)
thermal.plot(
    volumes * 1e3,
    processes.adiabatic_final_temperature(TEMPERATURE, V1, volumes, start.gamma),
    lw=2.2, color="#dc2626",
)
thermal.plot([V1 * 1e3], [TEMPERATURE], "ko", ms=8, zorder=6)
for route, colour in zip(
    (slow, loaded, free), ("#dc2626", "#d97706", "#059669"), strict=True
):
    thermal.plot([V2 * 1e3], [route.end.temperature], "o", ms=8, color=colour, zorder=6)
thermal.set_xlabel("volume (L)")
thermal.set_ylabel("temperature (K)")
thermal.set_title("only the slow route has a curve")

plt.tight_layout()
plt.show()

בקשת העקומה של הדרכים הבלתי הפיכות היא שגיאה, לא `None`:

In [ ]:
try:
    curve = free.quasistatic_path
except ValueError as error:
    print(f"ValueError: {error}")

## חלק 3 — ההתפשטות החופשית, רמה אחת למטה

הטיעון התרמודינמי אמר $\Delta U = 0$, ולכן $\Delta T = 0$. הנה אותה טענה בדיוק, ללא שמץ של
תרמודינמיקה.

קחו קופסת חלקיקים מן המודל הקינטי של מודול 4 והרחיבו אותה. זו כל הפעולה:
`free_expansion_microstate` מעתיקה את המהירויות ומגדילה את הקופסה. הסרת מחיצה אינה מזיזה
דופן כנגד כוח, ולכן שום מהירות של שום חלקיק אינה יכולה להשתנות — והטמפרטורה הקינטית היא
פונקציה של המהירויות בלבד.

אחר כך תנו לסימולציה לרוץ ומדדו את הלחץ, וזוהי מדידה אמיתית: התנע המצטבר שנמסר לדפנות, מחולק
בזמן ובמידת הדופן.

In [ ]:
ARGON_MASS = 4.65e-26  # kg
N_MICRO = 400
STEPS = 1500

rng = np.random.default_rng(2026)
gas = kinetics.initialise_gas(N_MICRO, (1.0e-6, 1.0e-6), 300.0, ARGON_MASS, rng)
widened = processes.free_expansion_microstate(gas, factor=VOLUME_RATIO)

before = kinetics.simulate(gas, dt=kinetics.max_stable_dt(gas), n_steps=STEPS)
after = kinetics.simulate(widened, dt=kinetics.max_stable_dt(widened), n_steps=STEPS)

# The widened run starts with every particle bunched in the old half, so the first crossing is
# a transient the equilibrium pressure has to be measured past.
p_before = before.pressure(discard_fraction=0.2)
p_after = after.pressure(discard_fraction=0.2)

print(f"temperature before  {gas.kinetic_temperature:.9f} K")
print(f"temperature after   {widened.kinetic_temperature:.9f} K")
print(f"ratio               {widened.kinetic_temperature / gas.kinetic_temperature:.12f}\n")
print(f"pressure before     {p_before:.4e} Pa")
print(f"pressure after      {p_after:.4e} Pa")
print(f"ratio               {p_after / p_before:.4f}   (expected {1 / VOLUME_RATIO:.4f})")

יחס הטמפרטורות אינו *בערך* אחד; הוא אחד עד הספרה האחרונה שיש למספר צף, והוא היה אחד עבור כל
זרע, כל מספר חלקיקים וכל גורם הרחבה. זו אינה מדידה — זו תכונה של הקוד, וזו התכונה הנכונה.
אין ב-`free_expansion_microstate` מקום שבו מהירות יכולה לקטון.

יחס הלחצים *כן* מדידה, ולכן הוא מגיע עם פיזור. הנה הפיזור בין זרעים בלתי תלויים:

In [ ]:
def measure_pressure_ratio(generator):
    """One microscopic free expansion, from its own generator: no global RNG anywhere."""
    initial = kinetics.initialise_gas(N_MICRO, (1.0e-6, 1.0e-6), 300.0, ARGON_MASS, generator)
    expanded = processes.free_expansion_microstate(initial, factor=VOLUME_RATIO)
    packed = kinetics.simulate(initial, dt=kinetics.max_stable_dt(initial), n_steps=1200)
    spread = kinetics.simulate(expanded, dt=kinetics.max_stable_dt(expanded), n_steps=1200)
    return spread.pressure(discard_fraction=0.2) / packed.pressure(discard_fraction=0.2)


study = seed_study(measure_pressure_ratio, n_seeds=6, base_seed=17)
print(f"pressure ratio across seeds: {study.mean:.4f} +/- {study.standard_error:.4f}")
print(f"agrees with 1/2 within 3 sigma: {study.agrees_with(0.5)}")
print(f"relative spread: {study.relative_spread:.4f}")

## חלק 4 — לחקור

שני כפתורים. `degrees_of_freedom` משנה את הגז ($3$ חד-אטומי, $5$ דו-אטומי, $6$ עם הרטט
פעיל), ו-`load_fraction` קובע את הלחץ החיצוני הקבוע כשבר מן העומס הגדול ביותר שעדיין מאפשר
לגז להגיע ל-$2V_1$ — כך ש-$1$ הוא הדרך עם העומס מחלק 2 ו-$0$ הוא ההתפשטות החופשית, במדויק
ולא בקירוב.

לחצו **Run Interact** אחרי הזזת מחוון. שני דברים שכדאי למצוא:

1. ערך של $f$ שעבורו האדיאבטה קרובה באופן נראה לעין לאיזותרמה. מה $f \to \infty$ היה אומר
   מבחינה פיזיקלית?
2. שבר העומס שבו הדרך הבלתי הפיכה מספקת מחצית מן העבודה שמספקת האיטית.

In [ ]:
import ipywidgets as widgets


def compare_routes(degrees_of_freedom=3, load_fraction=1.0):
    """Redraw the three adiabatic routes for a different gas and a different load."""
    state = processes.EquilibriumState.from_temperature(
        N_PARTICLES, TEMPERATURE, V1, degrees_of_freedom
    )
    ceiling = processes.external_pressure_for_equilibrium_at(state, VOLUME_RATIO)
    load = load_fraction * ceiling

    quasistatic = processes.adiabatic(state, V2)
    isotherm = processes.isothermal(state, V2)
    if load > 0:
        irreversible = processes.against_constant_external_pressure(state, load, V2)
    else:
        irreversible = processes.free_expansion(state, V2)

    path = quasistatic.quasistatic_path
    reference = isotherm.quasistatic_path
    fig, (plane, bars) = plt.subplots(
        1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1.5, 1.0]}
    )
    plane.plot(reference.volumes * 1e3, reference.pressures, "--", color="#059669", lw=1.6)
    plane.plot(path.volumes * 1e3, path.pressures, lw=2.2, color="#dc2626")
    plane.fill_between(path.volumes * 1e3, path.pressures, alpha=0.18, color="#dc2626")
    plane.axhline(load, color="#d97706", lw=1.4, ls=":")
    plane.fill_between(
        np.linspace(V1, V2, 100) * 1e3, load, alpha=0.20, color="#d97706"
    )
    plane.plot([V1 * 1e3], [state.pressure], "ko", ms=7, zorder=6)
    plane.set_xlabel("volume (L)")
    plane.set_ylabel("pressure (Pa)")
    plane.set_ylim(0, 1.12 * state.pressure)
    plane.set_title(f"gamma = {state.gamma:.3f},  load = {load / state.pressure:.3f} P1")

    finals = [quasistatic.end.temperature, irreversible.end.temperature, TEMPERATURE]
    bars.bar(range(3), finals, width=0.6, color=["#dc2626", "#d97706", "#059669"])
    bars.set_xticks(range(3))
    bars.set_xticklabels(["slow", "loaded", "free"])
    bars.set_ylabel("final temperature (K)")
    bars.set_ylim(0, 1.12 * TEMPERATURE)
    for index, value in enumerate(finals):
        bars.text(index, value + 6, f"{value:.0f}", ha="center", fontsize=9)

    plt.tight_layout()
    plt.show()

    # abs(), not a leading minus: work delivered by a free expansion is zero, and "-0.0000e+00"
    # reads like a bug rather than like nothing happening.
    print(f"work delivered  slow {abs(quasistatic.work_on_gas):.4e} J     "
          f"irreversible {abs(irreversible.work_on_gas):.4e} J")


widgets.interact_manual(
    compare_routes,
    degrees_of_freedom=widgets.IntSlider(min=3, max=8, step=1, value=3, description="f"),
    load_fraction=widgets.FloatSlider(
        min=0.0, max=1.0, step=0.05, value=1.0, description="load"
    ),
);

## חלק 5 — בדיקות אוטומטיות

אותן טענות רצות גם בערכת המבחנים של הפרויקט, כך שהטענות בעמוד המודול אינן יכולות להירקב
בשקט. שימו לב שכל אנרגיה כאן היא בסדר גודל של $10^{-18}\,\mathrm{J}$, הרחק בתוך הסבילות
המוחלטת שנומפיי מניחה כברירת מחדל — ולכן כל השוואה שלהלן היא יחסית. שימוש תמים ב-`np.isclose`
היה מדווח ששלוש הדרכים האדיאבטיות ביצעו אותה עבודה.

In [ ]:
# 1. Each family's sampled path reproduces its own closed form under the trapezoid rule.
for family in families:
    quadrature = family.quasistatic_path.work_on_gas()
    if family.work_on_gas == 0.0:
        assert abs(quadrature) < 1e-30, family.label
    else:
        assert relative_error(quadrature, family.work_on_gas) < 1e-4, family.label

# 2. The first law closes on every process, quasistatic or not. For the isobaric row this is a
#    live check of Mayer's relation: W_on and Q were computed by independent formulae.
for _, route in routes:
    scale_j = max(abs(route.work_on_gas), abs(route.internal_energy_change), 1e-30)
    assert abs(route.first_law_residual) / scale_j < 1e-12, route.label
for family in families:
    scale_j = max(abs(family.work_on_gas), abs(family.internal_energy_change), 1e-30)
    assert abs(family.first_law_residual) / scale_j < 1e-12, family.label

# 3. Both adiabatic invariants are flat along the whole curve, not just at its ends.
adiabatic_path = adiabat.quasistatic_path
pv_gamma = adiabatic_path.pressures * adiabatic_path.volumes ** start.gamma
temperatures = adiabatic_path.pressures * adiabatic_path.volumes / (N_PARTICLES * 1.380649e-23)
tv_gamma = temperatures * adiabatic_path.volumes ** (start.gamma - 1.0)
assert np.ptp(pv_gamma) / pv_gamma.mean() < 1e-12
assert np.ptp(tv_gamma) / tv_gamma.mean() < 1e-12

# 4. The three adiabatic routes are ordered, and the free expansion is exactly isothermal.
assert slow.end.temperature < loaded.end.temperature < free.end.temperature
assert abs(slow.work_on_gas) > abs(loaded.work_on_gas) > 0.0
assert free.work_on_gas == 0.0  # a literal in the constructor, not a computed result
assert relative_error(free.end.internal_energy, start.internal_energy) < 1e-15
assert relative_error(slow.work_on_gas, loaded.work_on_gas) > 0.25

# 5. The microscopic free expansion changes no velocity, and halves the measured pressure.
assert widened.kinetic_temperature == gas.kinetic_temperature
assert study.agrees_with(0.5)

print(f"quadrature error on the adiabat: "
      f"{relative_error(adiabatic_path.work_on_gas(), adiabat.work_on_gas):.2e}")
print(f"P V^gamma spread along the curve: {np.ptp(pv_gamma) / pv_gamma.mean():.2e}")
print(f"final temperatures: {slow.end.temperature:.1f} < "
      f"{loaded.end.temperature:.1f} < {free.end.temperature:.1f} K")
print("\nall checks passed")

## בדקו את ההבנה שלכם

הריצו את התא שלהלן לחידון עם בדיקה אוטומטית.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "06-processes.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

1. גם ההתפשטות החופשית וגם האיזותרמית מעבירות את הגז הזה מ-$V_1$ ל-$2V_1$ עם $\Delta T = 0$.
   מנו כל גודל שחושב במחברת הזו שמבדיל ביניהן.
2. יחס הטמפרטורות בחלק 3 היה $1$ בדיוק ויחס הלחצים היה $0.50 \pm 0.01$. אחד משני המספרים
   האלה הוא ראיה והשני אינו. אמרו איזה, ומדוע.
3. לא יכולתם לבקש מן הספרייה את המסלול של ההתפשטות החופשית — היא זרקה שגיאה במקום להחזיר
   כלום. הסבירו מה שרטוט שנבנה על `None` שהוחזר בשקט היה מראה, ומדוע זריקת שגיאה היא ההתנהגות
   הכנה.

**התשובות שלכם:**

1.
2.
3.